# Explainable ML-Based Malicious DNS Traffic Detection
> Predictions are estimates, not proof. Offline feature-CSV tool. Primary dataset CIC-Bell-DNS-EXF-2021 (gated download). This notebook runs on real CSV if present, else clearly-synthetic demo data.

In [ ]:
# Cell 0 - versions
import sys, platform
import pandas as pd, numpy as np, sklearn, joblib, matplotlib
import xgboost, shap
print('python', sys.version.split()[0])
for m,n in [(pd,'pandas'),(np,'numpy'),(sklearn,'sklearn'),(xgboost,'xgboost'),(shap,'shap')]: print(n, m.__version__)

## 1. Get data
Official: https://www.unb.ca/cic/datasets/dns-exf-2021.html - complete the form, download, upload CSVs to Colab /content or Drive. Or run synthetic demo below (clearly synthetic).

In [ ]:
import glob, os
print(glob.glob('/content/*.csv')[:10])
# Drive option:
# from google.colab import drive; drive.mount('/content/drive')
from src.synthetic import make_synthetic
import pandas as pd
try:
    f=sorted(glob.glob('data/*.csv'))
    real=[c for c in f if 'synthetic' not in c]
    df=pd.read_csv(real[0]) if real else make_synthetic(5000,42)
    print('SOURCE:', real[0] if real else 'SYNTHETIC DEMO ONLY', df.shape)
except Exception as e:
    print(e)

## 2. Audit: shape, columns, dtypes, labels, missing, duplicates, inf

In [ ]:
from src.data_loader import basic_summary, candidate_label_info, detect_target_column
import json
print(json.dumps(basic_summary(df), indent=2)[:3000])
print('TARGET GUESS:', detect_target_column(df))
print(json.dumps(candidate_label_info(df), indent=2)[:4000])
print(df.head(3).to_string())

## 3. Preprocessing + split (test untouched, leakage flags)

In [ ]:
from src.validation import data_quality_report, select_features
from src.preprocessing import clean_frame
rep=data_quality_report(df)
print('leakage hits:', rep['leakage_hits'])
print('constant:', rep['constant_columns'])
TARGET=detect_target_column(df) or 'label'
feats=select_features(df, TARGET)
print(len(feats), feats[:10])

## 4-5. Train LR/RF/XGB binary + multiclass, evaluate

In [ ]:
from src.train import train_task
from src.evaluate import comparison_table
for task in (['binary','multiclass'] if df[TARGET].nunique()>=3 else ['binary']):
    res=train_task(df, TARGET, task)
    print(task); print(comparison_table(res).to_string())

## 6. SHAP (sampled)

In [ ]:
from src.explain import shap_global_local, top_features
import joblib
# after scripts/train_all.py, load artifact; here demo on fresh train
res=train_task(df, TARGET, 'binary')
r=shap_global_local(res['models']['xgb']['pipeline'], clean_frame(df[res['features']]).head(400), explain_n=100)
print(top_features(r['shap_values'], r['feature_names'], 8))

## 7. Save + reload pipeline
See scripts/train_all.py for full save. Never save unfitted transformers.

In [ ]:
from src.train import save_artifact
import yaml
cfg=yaml.safe_load(open('configs/model_config.yaml'))
# example: save_artifact(res['models']['xgb']['pipeline'], res['features'], res['label_info']['classes'], 'binary', res['models']['xgb']['eval'], cfg, 'models/pipeline_binary.joblib')
print('see scripts/train_all.py')